In [ ]:
import torch
import pprint

# file_path = '/root/private_data/luog/codex/IgGM/data/sabdab_zip/processed/sabdab_debug/samples/5yy5_C_D_B.pt'
# file_path = '/root/private_data/luog/codex/IgGM/data/sabdab_2603/processed/sabdab_2603/samples/2htk_C_D_A.pt'

file_path = "/root/private_data/luog/codex/IgGM/data/sabdab_2603/processed/sabdab_2603/samples/7yzj_H_L_E.pt"

try:
    # 加载数据，map_location='cpu' 确保在没有 GPU 的环境下也能加载
    data = torch.load(file_path, map_location='cpu')
    # print(data['antibody_region'].keys())
    
    print(f"--- 文件加载成功：{file_path} ---")
    print(f"数据类型：{type(data)}")
    
    # 根据数据类型不同，展示内容
    if isinstance(data, dict):
        print("\n[字典键值]:")
        for key, value in data.items():
            if hasattr(value, 'shape'):
                print(f"  Key: {key}, Type: {type(value)}, Shape: {value.shape}")
            else:
                print(f"  Key: {key}, Type: {type(value)}, Value: {value}")
                
        # 尝试打印第一个 key 的部分内容
        if len(data) > 0:
            first_key = list(data.keys())[0]
            first_val = data[first_key]
            if isinstance(first_val, torch.Tensor):
                print(f"\n[示例数据 - Key '{first_key}' 的前 5 行]:")
                print(first_val[:5])
                
    elif isinstance(data, torch.Tensor):
        print(f"\n[Tensor 信息]:")
        print(f"  Shape: {data.shape}")
        print(f"  Dtype: {data.dtype}")
        print(f"  前 5 行内容:\n{data[:5]}")
        
    elif isinstance(data, list) or isinstance(data, tuple):
        print(f"\n[列表/元组长度]: {len(data)}")
        if len(data) > 0:
            print(f"  第一个元素类型：{type(data[0])}")
            if hasattr(data[0], 'shape'):
                print(f"  第一个元素形状：{data[0].shape}")
    
    else:
        print("\n[内容]:")
        print(data)

except Exception as e:
    print(f"加载文件时出错：{e}")
    print("请确保已安装 PyTorch (pip install torch)")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import torch
from pathlib import Path

def scan_pt_files(directory, threshold=1200):
    dir_path = Path(directory)
    long_files = []
    pt_files = list(dir_path.glob("*.pt"))
    
    for pt_file in pt_files:
        # 加载.pt 文件
        data = torch.load(pt_file, map_location='cpu')
        data_len = data['sequence_lengths']['H']+data['sequence_lengths']['L']+data['sequence_lengths']['A']

        print(f"文件: {pt_file.name}, sum_hla: {data_len}"   )
   

if __name__ == "__main__":
    # 目标目录
    target_dir = "/root/private_data/luog/codex/IgGM/data/sabdab/processed/sabdab/samples"
    
    # 执行扫描
    long_files = scan_pt_files(target_dir, threshold=1200)


In [ ]:

a = "QCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVATPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTPSALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFK"
len(a)

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from tqdm import tqdm

def scan_pt_files(directory, threshold=1200):
    """扫描 .pt 文件并收集序列长度数据"""
    dir_path = Path(directory)
    data_lengths = []  # 收集所有长度
    pt_files = list(dir_path.glob("*.pt"))
    
    print(f"开始扫描 {len(pt_files)} 个文件...")
    
    for idx, pt_file in tqdm(enumerate(pt_files)):
        try:
            # 加载.pt 文件
            data = torch.load(pt_file, map_location='cpu')
            if 'A' in data['sequence_lengths']:
                if 'L' in data['sequence_lengths'] and 'H' in data['sequence_lengths']:
                    data_len = (data['sequence_lengths']['H'] + 
                            data['sequence_lengths']['L'] + 
                            data['sequence_lengths']['A'])
                else:
                    if 'H' in data['sequence_lengths']:
                        data_len = (data['sequence_lengths']['H'] + 
                                data['sequence_lengths']['A'])
                    elif 'L' in data['sequence_lengths']:
                        data_len = (data['sequence_lengths']['L'] + 
                                data['sequence_lengths']['A'])
                    
            data_lengths.append(int(data_len))
            
        except Exception as e:
            continue
    
    return data_lengths


def plot_length_distribution(data_lengths, threshold=1200, save_path="length_distribution.png"):
    data_lengths = np.array(data_lengths)
    
    # 创建 2x2 子图布局
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('序列长度统计分布分析', fontsize=16, fontweight='bold')
    
    # === 图1: 直方图 + 阈值线 ===
    ax1 = axes[0, 0]
    ax1.hist(data_lengths, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
    ax1.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'阈值: {threshold}')
    ax1.axvline(data_lengths.mean(), color='orange', linestyle=':', linewidth=2, 
                label=f'均值: {data_lengths.mean():.1f}')
    ax1.set_xlabel('序列总长度 (H+L+A)')
    ax1.set_ylabel('文件数量')
    ax1.set_title('长度分布直方图')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # === 图2: KDE 密度曲线 ===
    ax2 = axes[0, 1]
    kde = stats.gaussian_kde(data_lengths)
    x_range = np.linspace(data_lengths.min(), data_lengths.max(), 200)
    ax2.plot(x_range, kde(x_range), color='darkblue', linewidth=2)
    ax2.fill_between(x_range, kde(x_range), alpha=0.3, color='lightblue')
    ax2.axvline(threshold, color='red', linestyle='--', linewidth=1.5, label=f'阈值: {threshold}')
    ax2.set_xlabel('序列总长度')
    ax2.set_ylabel('概率密度')
    ax2.set_title('核密度估计 (KDE)')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # === 图3: 阈值占比饼图 ===
    ax3 = axes[1, 0]
    below = np.sum(data_lengths < threshold)
    above = np.sum(data_lengths >= threshold)
    total = len(data_lengths)
    sizes = [below, above]
    labels = [f'< {threshold}\n({below/total*100:.1f}%)', 
              f'≥ {threshold}\n({above/total*100:.1f}%)']
    colors = ['lightgreen', 'lightcoral']
    ax3.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', 
            startangle=90, textprops={'fontsize': 10})
    ax3.set_title(f'阈值 {threshold} 占比分布')
    ax3.axis('equal')
    
    # === 图4: 箱线图 + 统计信息 ===
    ax4 = axes[1, 1]
    bp = ax4.boxplot([data_lengths], vert=True, patch_artist=True, 
                     labels=['序列长度'], widths=0.5)
    bp['boxes'][0].set_facecolor('lightblue')
    ax4.axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'阈值: {threshold}')
    
    # 添加统计文本框
    stats_text = (f'样本数: {total}\n'
                  f'均值: {data_lengths.mean():.1f}\n'
                  f'中位数: {np.median(data_lengths):.1f}\n'
                  f'标准差: {data_lengths.std():.1f}\n'
                  f'最小: {data_lengths.min()}\n'
                  f'最大: {data_lengths.max()}')
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.3)
    ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes, fontsize=9,
             verticalalignment='top', bbox=props)
    
    ax4.set_ylabel('序列长度')
    ax4.set_title('箱线图 + 统计摘要')
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)
    
    # 调整布局并保存
    plt.tight_layout()
    # plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"📊 统计图已保存至: {save_path}")
    plt.show()
    
    # 打印关键统计信息
    print(f"\n📈 关键统计:")
    print(f"   总文件数: {total}")
    print(f"   平均长度: {data_lengths.mean():.2f} ± {data_lengths.std():.2f}")
    print(f"   中位数: {np.median(data_lengths)}")
    print(f"   < {threshold}: {below} 个 ({below/total*100:.1f}%)")
    print(f"   ≥ {threshold}: {above} 个 ({above/total*100:.1f}%)")


if __name__ == "__main__":
    # 配置参数
    target_dir = "/root/private_data/luog/codex/IgGM/data/sabdab/processed/sabdab/samples"
    threshold = 1200  # 长度阈值
    output_fig = "sabdab_length_distribution.png"  # 输出图片路径
    
    # 1. 扫描并收集数据
    data_lengths = scan_pt_files(target_dir, threshold=threshold)
    
    # 2. 绘制分布图
    if data_lengths:
        plot_length_distribution(data_lengths, threshold=threshold, save_path=output_fig)
    else:
        print("❌ 未收集到有效数据，跳过绘图")

In [ ]:
import torch
import gc

# 检查是否使用 ROCm
if torch.cuda.is_available():
    # 清理缓存
    torch.cuda.empty_cache()
    
    # 垃圾回收
    gc.collect()
    
    # 重置内存统计
    torch.cuda.reset_max_memory_allocated()
    torch.cuda.reset_max_memory_cached()
    
    print(f"已分配：{torch.cuda.memory_allocated()/1024**2:.2f} MB")
    print(f"缓存：{torch.cuda.memory_reserved()/1024**2:.2f} MB")

In [ ]:
# read_ids.py
def read_sample_ids(filepath):
    """读取 txt 文件，返回 ID 列表"""
    with open(filepath, 'r', encoding='utf-8') as f:
        # 去除空行和首尾空格
        ids = [line.strip() for line in f if line.strip()]
    return ids

# 使用
ids = read_sample_ids('/root/private_data/luog/codex/IgGM/data/sabdab/processed/sabdab_file/split/train_prot_ids.txt')
print(f"共读取 {len(ids)} 个样本")
for i in ids:
    all = i.split("_")
    if all[1]=="NA" or all[-1]=="NA":
        print()
# print(ids[:5])  # 预览前 5 个

In [ ]:


from IgGM.protein.prot_constants import RESD_NAMES_1C,restype_atom14_to_atom37
print(restype_atom14_to_atom37)

In [ ]:
from openfold.np import residue_constants
print(residue_constants.RESTYPE_ATOM14_TO_ATOM37)
print(residue_constants.atom_order)

In [ ]:

from collections import OrderedDict
RESD_MAP_1TO3 = OrderedDict([
    ('A', 'ALA'),
    ('R', 'ARG'),
    ('N', 'ASN'),
    ('D', 'ASP'),
    ('C', 'CYS'),
    ('Q', 'GLN'),
    ('E', 'GLU'),
    ('G', 'GLY'),
    ('H', 'HIS'),
    ('I', 'ILE'),
    ('L', 'LEU'),
    ('K', 'LYS'),
    ('M', 'MET'),
    ('F', 'PHE'),
    ('P', 'PRO'),
    ('S', 'SER'),
    ('T', 'THR'),
    ('W', 'TRP'),
    ('Y', 'TYR'),
    ('V', 'VAL')
])

RESD_MAP_3TO1 = {v: k for k, v in RESD_MAP_1TO3.items()}

# note that order not compact with alphafold
RESD_NAMES_1C = sorted(list(RESD_MAP_1TO3.keys()))
RESD_NAMES_1C

NameError: name 'np' is not defined

In [2]:
import os

target = "TIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITL"

def search_fasta(folder):
    for root, _, files in os.walk(folder):
        for file in files:
            if file.endswith(".fasta") or file.endswith(".fa"):
                path = os.path.join(root, file)

                with open(path, "r") as f:
                    header = None
                    seq = ""

                    for line in f:
                        line = line.strip()
                        if line.startswith(">"):
                            # 检查上一条序列
                            if header and target in seq:
                                print(f"\n=== Found in {path} ===")
                                print(header)
                                print(seq)

                            header = line
                            seq = ""
                        else:
                            seq += line

                    # 最后一条序列也要检查
                    if header and target in seq:
                        print(f"\n=== Found in {path} ===")
                        print(header)
                        print(seq)


# 使用
search_fasta("/root/private_data/luog/codex/IgGM/data/sabdab_2603/processed/fasta")

len("SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTL")


=== Found in /root/private_data/luog/codex/IgGM/data/sabdab_2603/processed/fasta/3ogo_F_NA_C.fasta ===
>A
SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITL


63